# Shallow diffusion - choosing how much diffusion may change the speech
Full diffusion (strength 1.0) starts from pure noise and can invent wrong sounds. Shallow diffusion (DiffSinger, 2022) starts from the FastSpeech mel with a little noise added, so the words come from FastSpeech and diffusion only sharpens detail.

- **strength 0** = no diffusion (FastSpeech mel straight to the vocoder)
- **strength 1** = the old full diffusion

No retraining: this uses the diffusion model you just trained. **Restart the kernel**, then run the cells in order.

In [1]:
import sys, torch
ROOT = r"C:\Users\INTEL\Downloads\voice_cloning_v2_1\voice_cloning_v2_1"
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - select the Python 3.11.9 kernel")

GPU: NVIDIA GeForce RTX 4070


In [2]:
import numpy as np
from IPython.display import Audio, display
from vc_lib.infer import VoicePipeline, evaluate
from vc_lib.common import SR

pipe = VoicePipeline(ROOT, diffusion_steps=50, cfg=1.0)
spk = pipe.meta['train_speakers'][0]; unseen = pipe.meta['holdout'][0]
sentences = ['Please call Stella.', 'The weather is nice today.', 'I will send the report tomorrow morning.',
             'Can you repeat that, please?', 'Your order has been shipped.',
             'The meeting has been moved to Thursday afternoon.',     # new text, not in VCTK
             'Your package will arrive between nine and eleven.']     # new text, not in VCTK

Text-to-mel model: fastspeech | diffusion: on (strength 0.3)


## Measure each strength (about 2-3 minutes)
Lower WER = clearer words. Higher similarity = closer to the target voice.

In [3]:
import io, contextlib
STRENGTHS = [0.0, 0.1, 0.2, 0.3, 0.5, 1.0]
results = {}
for s in STRENGTHS:
    pipe.strength = s
    with contextlib.redirect_stdout(io.StringIO()):          # keep the output short
        w, sim = evaluate(pipe, sentences, speaker=spk)
    results[s] = (np.mean(w), np.mean(np.array(w) == 0), np.mean(sim))
    print(f'strength {s:.1f}:  WER {results[s][0]:.3f}  | sentences perfect {results[s][1]:.0%}  | similarity {results[s][2]:.3f}')
best = min(STRENGTHS, key=lambda s: (round(results[s][0], 3), -results[s][2]))
print(f'\nLowest WER: strength {best}')

c:\Users\INTEL\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


strength 0.0:  WER 0.083  | sentences perfect 71%  | similarity 0.779
strength 0.1:  WER 0.083  | sentences perfect 71%  | similarity 0.744
strength 0.2:  WER 0.124  | sentences perfect 57%  | similarity 0.731
strength 0.3:  WER 0.171  | sentences perfect 43%  | similarity 0.723
strength 0.5:  WER 0.153  | sentences perfect 43%  | similarity 0.715
strength 1.0:  WER 0.417  | sentences perfect 29%  | similarity 0.709

Lowest WER: strength 0.0


## Listen: same sentence at each strength
WER only checks the words. Listen for which sounds most natural and least metallic.

In [4]:
text = 'The meeting has been moved to Thursday afternoon.'
for s in STRENGTHS:
    pipe.strength = s
    print(f'strength {s}')
    display(Audio(pipe.synthesize(text, speaker=spk), rate=SR))

strength 0.0


strength 0.1


strength 0.2


strength 0.3


strength 0.5


strength 1.0


## Details for one strength
Set `s` to the strength you want to inspect (for example the best one above).

In [5]:
s = best
pipe.strength = s
print(f'--- strength {s}, training voice ---'); evaluate(pipe, sentences, speaker=spk)
print(f'--- strength {s}, unseen voice ---'); evaluate(pipe, sentences, speaker=unseen)

--- strength 0.0, training voice ---
WER 0.33 | similarity 0.54 | 'Please call Stella.' -> 'Please call staff'
WER 0.00 | similarity 0.86 | 'The weather is nice today.' -> 'The weather is nice today.'
WER 0.00 | similarity 0.84 | 'I will send the report tomorrow morning.' -> 'I will send the report tomorrow morning.'
WER 0.00 | similarity 0.73 | 'Can you repeat that, please?' -> 'Can you repeat that, please?'
WER 0.00 | similarity 0.83 | 'Your order has been shipped.' -> 'Your order has been shipped.'
WER 0.00 | similarity 0.81 | 'The meeting has been moved to Thursday afternoon.' -> 'The meeting has been moved to Thursday afternoon.'
WER 0.25 | similarity 0.86 | 'Your package will arrive between nine and eleven.' -> 'your package will arrive between that and'

Mean WER: 0.083 (lower is better)   Mean speaker similarity: 0.779
--- strength 0.0, unseen voice ---
WER 0.33 | similarity 0.46 | 'Please call Stella.' -> 'Please call stuff'
WER 0.00 | similarity 0.56 | 'The weather is nice to

([0.3333333333333333, 0.0, 0.0, 0.0, 0.0, 0.25, 0.0],
 [0.4647035598754883,
  0.5560676455497742,
  0.6546005010604858,
  0.4864550828933716,
  0.6456064581871033,
  0.6369601488113403,
  0.6197085380554199])

## Save audio for your report

In [6]:
import soundfile as sf, os
out = os.path.join(ROOT, 'demo_outputs'); os.makedirs(out, exist_ok=True)
for s in [0.0, best, 1.0]:
    pipe.strength = s
    sf.write(os.path.join(out, f'meeting_strength_{s}.wav'), pipe.synthesize(text, speaker=spk), SR)
print('saved to', out)

saved to C:\Users\INTEL\Downloads\voice_cloning_v2_1\voice_cloning_v2_1\demo_outputs
